
# Cleaning Dataset 

Perform data cleaning on the SYNCH Rapid City Dataset, including formating the variable names, spell checking. 



In [1]:
%load_ext autoreload
%autoreload 2

# Import library

In [12]:

from sentence_transformers import SentenceTransformer, util , SimilarityFunction
from pathlib import Path


import pandas as pd
import numpy as np
import regex as re
import itertools as it
import matplotlib.pyplot as plt
import pickle
import os
import torch
import seaborn as sns 
import sys
import json


In [15]:

# Get the current working directory
cwd = Path.cwd()

# Get the path to the parent directory
parent_dir = cwd.parent

# Add the parent directory to the Python path
sys.path.append(str(parent_dir))

# Now you can import the data_io module from src
import src.data_io as data_io

# Read data


## Read Raw data

In [10]:
print(os.getcwd())

c:\Users\GL665\Documents\Phrame-MentalMaps\notebook


In [17]:
# Go one level up from the current directory
cwd = os.getcwd()

parent_dir = Path(cwd).parent
print(parent_dir)

my_data_RC = {}
folder_path = str(parent_dir / 'results/SYNCH/JSON_file/RC/')


cld_data = data_io.read_json(folder_path, my_data_RC)

c:\Users\GL665\Documents\Phrame-MentalMaps
load: C 7 6 20 clean model.json
load: drug justice model clean.json
load: DSS23.json
load: DV justice model clean.json
load: E clean model 572020.json
load: FC Prov Clean.json
load: FM 51220Clean.json
load: full circle clean.json
load: full circle GBM 5 24 21clean.json
load: nat model clean2.json
load: Passages Clean.json
load: reentry clean model.json
load: Wavi combined clean.json


## Read already preprocessed data

In [ ]:
# read the data 
# file_path = "results/SYNCH/JSON_file/RC_cld_data_cleaned_preprocessed.json"

# # Pickle dump the cld_data_new dictionary
# with open(file_path, "rb") as file:
#     cld_data = pickle.load(file)

# print(cld_data)



# Cleaning Vairable Names 

## Clean the format of the vairable names 

In [18]:
def clean_variable_name(variable_name):
    """
    Cleans a variable name by performing the following transformations:
    - Replaces '&' with 'and'
    - Replaces '/' with ' and '
    - Replaces '_' with ' '
    - Replaces common abbreviations and acronyms with their full forms
    - Removes leading/trailing whitespace
    - Removes extra whitespace between words
    - Converts the name to lowercase
    
    Args:
        variable_name (str): The variable name to be cleaned.
    
    Returns:
        str: The cleaned variable name.
    """
        
    replacements = {
        # TODO: Q: Need to refine these. 
        '&': 'and',  # remove '&'
        '/': ' and ',  # replace '/' with ' or '
        # '-': ' ',  # replace '-' with ' '
        '_': ' ',  # replace '_' with ' '
        'ipv': 'intimate partner violence',
        'dss': ' department of social service',
        'aian': 'american indian and alaska native',
        'u.s.': 'U.S.',
        'passages': 'reentry facility', #?
        'fasd':'Fetal alcohol spectrum disorder',
        'alc m': 'alcohol mortality',
        'mh': 'mental health',
        'tx': 'tax',
        'ihs': 'indian health service',
        'med': 'medication',
        'sub': 'substance',
        'ptsd': 'Post-traumatic stress disorder',
        'aep':'anti-establishment parties',
        'skillset':'skill set',
        'rez':'restorative',
        'aces': 'adverse childhood experience', 
        'ai': 'american indian'

    }

    # TODO: Feedback: Read the CLDs again once the renaming is fixed. 

    for old, new in replacements.items():
        variable_name = variable_name.replace(old, new)

    variable_name = variable_name.strip()  # remove leading/trailing whitespace
    variable_name = " ".join(variable_name.split())  # remove extra whitespace between words
    variable_name = variable_name.lower()  # convert to lowercase

    return variable_name


In [19]:
# TODO: Q: Need to refine these 
custom_words = ['american indian and alaska native', 
                'indian', 
                'alaska',
                'christianity', 
                'fostercare', 
                'american',
                'child-rearing',
                'reoffending', 
                'lakota',
                'mitakuye oyasin', 
                'pushback',
                'prosocial']

#TODO: switch aian to indigenous 

# clean variable format 
i = 0
variable_clean_dict = {}
for k, v in cld_data.items():
    for key, value in v[0].items():
        
        correct_value = clean_variable_name(value)
        if value != correct_value: 
            variable_clean_dict[value]= correct_value
            i += 1

        v[0][key] = correct_value    
        


print('---')
print("Number of variables that corrected due to formating issue :", i)


---
Number of variables that corrected due to formating issue : 291


In [20]:
# with open("2_clean_format_dictionary.txt", "w") as fp:
#     json.dump(variable_clean_dict, fp) 


In [21]:
# write dictionary to a excel sheet, 
# with the original variable name and the corrected variable name
df_1 = pd.DataFrame(variable_clean_dict.items(), columns=['Original', 'Corrected'])


## Spell Checking 

In [22]:

from spellchecker import SpellChecker

def correct_spelling(text, custom_words = []):
    spell = SpellChecker()
    spell.word_frequency.load_words(custom_words)
    
    words = text.split()
    corrected_words = [spell.correction(word) or word for word in words]
    return " ".join(corrected_words)


In [23]:
i = 0
variable_correct_spell = {}
for k, v in my_data_RC.items():
    for key, value in v[0].items():
        corrected_text = correct_spelling(value, custom_words)
        if value != corrected_text: 
            variable_correct_spell[value] = corrected_text
            # print(f'Original: {value}')
            # print(f'Corrected: {corrected_text}')
            i += 1
        
        v[0][key] = corrected_text    
print('---')
print("Number of variables found needed spell check:", i)

---
Number of variables found needed spell check: 202


In [24]:
# with open("3_spell_check_dictionary.txt", "w") as fp:
#     json.dump(variable_correct_spell, fp) 

In [25]:
df_2 = pd.DataFrame(variable_correct_spell.items(), columns=['Original', 'Corrected'])

In [26]:

# with pd.ExcelWriter('results/SYNCH/file_rename/RC_variables_matching060424.xlsx') as writer:
# # Adding dataframes to Excel as new sheets
#     df_1.to_excel(writer, sheet_name='1.clean format', index=False)
#     df_2.to_excel(writer, sheet_name='2.spellcheck', index=False)


# Find semantic similary variable names using pretrained sentence transformer

In [27]:
# variable_clean_dict = {}
# for k, v in cld_data.items():
#     for key, value in v[0].items():
        
#         correct_value = clean_variable_name(value)
#         if value != correct_value: 
#             variable_clean_dict[value]= correct_value
#             i += 1

#         v[0][key] = correct_value    
        

In [28]:
variable_names = []

for k, v in cld_data.items():
    cld_new_vars_name = []
    cld_var_names = list(v[0].values())

    variable_names.extend(cld_var_names)


print("number of originial nodes: ", len(variable_names))

variable_names = np.sort(list(set(variable_names)))
print("number of originial unique nodes: ", len(variable_names))


number of originial nodes:  1166
number of originial unique nodes:  1019


In [29]:
# print the first 10 words of variable names
print(variable_names[:10])

['abandonment' 'ability for health insurance to pay for care'
 'ability for victims to receive housing resources'
 'ability to address trauma' 'ability to afford birth control'
 'ability to be self sustamerican inspanning'
 'ability to fulfil department of social service requests'
 'ability to fulfill court requirements'
 'ability to move from detox to treatment'
 'ability to navigate colonial adverse childhood experience']


In [41]:
# Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-mpnet-base-v2")

embeddings = model.encode(variable_names)

similarities = model.similarity(embeddings, embeddings)



c:\Users\GL665\Documents\Phrame-MentalMaps\venv\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [42]:
# Output the pairs with their score
# for idx_i, sentence1 in enumerate(variable_names):
#     print(sentence1)
#     for idx_j, sentence2 in enumerate(variable_names):
#         if similarities[idx_i][idx_j] > 0.75:
#             print(f" - {sentence2: <30}: {similarities[idx_i][idx_j]:.4f}")
#             break 

In [43]:
#TODO: see the fine tuning notebook
# TODO: set the threshold based on differnt perfromance 
# Threshold for considering names as similar
threshold = 0.75

# Group semantically similar names
groups = []
visited = set()

for idx, name in enumerate(variable_names):
    if name not in visited:
        similar_names = [name]
        for jdx, similarity in enumerate(similarities[idx]):
            if jdx != idx and similarity >= threshold and variable_names[jdx] not in visited:
                similar_names.append(variable_names[jdx])
                visited.add(variable_names[jdx])
                
        groups.append(similar_names)
        visited.update(similar_names)


In [44]:
print("numbers of final variables after grouping: ", len(groups))


numbers of final variables after grouping:  685


In [45]:
# save groups to excel sheet
df_merge_groups = pd.DataFrame(groups)
# df_merge_groups.to_excel('results/SYNCH/file_rename/RC_variables_grouped060424.xlsx', index=False)

In [46]:

# Print grouped variable names

for i, group in enumerate(groups):
    if len(group) > 1:
        print(f"Group {i + 1}: {group}")

len(groups)

Group 20: ['access to getting prenatal care', 'access to reproductive healthcare']
Group 37: ['affordability of treatment', 'affordability to medicationical care']
Group 46: ['american indian and alaska native mistrust of white medicationical staff', 'white medication and american indian and alaska native client racial tension']
Group 52: ['american indianlability of mental health therapy', 'american indianlable mental health in-jamerican indian resources']
Group 61: ['american indianlability of victim shelter housing', 'american indianlability of victim support and resources']
Group 63: ['american indianntamerican inspanning sober adverse childhood experience', 'engagement with post-treatment sobriety american indianntamerican indiannence']
Group 100: ['child physical violence', 'child sexual violence']
Group 105: ['child removal from home', 'kids removed from house']
Group 106: ['child removed from maternal custody', 'needing to get child back']
Group 132: ['communication with casewo

685

In [47]:
# create a dictionary to swap the words that are bundled in the group. 
swap_dict = {}

for group in groups:
    if len(group) > 1:
        longest_name = max(group, key=len)
        swap_dict[longest_name] = group
swap_dict

swap_dict_reversed = {v: k for k, vv in swap_dict.items() for v in vv}
swap_dict_reversed


{'access to getting prenatal care': 'access to reproductive healthcare',
 'access to reproductive healthcare': 'access to reproductive healthcare',
 'affordability of treatment': 'affordability to medicationical care',
 'affordability to medicationical care': 'affordability to medicationical care',
 'american indian and alaska native mistrust of white medicationical staff': 'white medication and american indian and alaska native client racial tension',
 'white medication and american indian and alaska native client racial tension': 'white medication and american indian and alaska native client racial tension',
 'american indianlability of mental health therapy': 'american indianlable mental health in-jamerican indian resources',
 'american indianlable mental health in-jamerican indian resources': 'american indianlable mental health in-jamerican indian resources',
 'american indianlability of victim shelter housing': 'american indianlability of victim support and resources',
 'american 

In [37]:
len(swap_dict_reversed)

480

In [48]:
# Swap the variable names in the data

cld_data_new = cld_data.copy()

for k, v in cld_data_new.items():
    for key, value in v[0].items():
        if value in swap_dict_reversed:
            v[0][key] = swap_dict_reversed[value]



In [49]:
variable_names = [list(v[0].values()) for k, v in cld_data_new.items()]

variable_names = sum(variable_names, [])

variable_names = np.sort(list(set(variable_names)))


print("number of originial unique nodes after cleaning the data: ", len(variable_names))


number of originial unique nodes after cleaning the data:  685


# write out the cleaned data to pickle

In [59]:
# # Specify the file path where you want to save the pickled data
file_path = "results/SYNCH/JSON_file/RC_cld_data_cleaned_and_merged.json"

merged_file_path = str(parent_dir / file_path)

# Pickle dump the cld_data_new dictionary
with open(merged_file_path, "wb") as file:
    pickle.dump(cld_data_new, file)


c:\Users\GL665\Documents\Phrame-MentalMaps\results\SYNCH\JSON_file\RC_cld_data_cleaned_and_merged.json
